# Preprocessing Handlers Utilities Demo

Demostración interactiva de las funciones de gestión y depuración temporal, ruido de sensores y tratamiento de valores ausentes implementadas en `_preprocessing_handlers.py` y expuestas por la fachada `ds_utils.preprocessing.preprocessing_utilities`.

Funciones cubiertas en este notebook:
- `handle_duplicates`
- `handle_time_split`
- `handle_time_bursts`
- `handle_sensor_noise`
- `handle_missing_values`

## Imports

In [1]:
# RECARGA AUTOMÁTICA DE MÓDULOS EXTERNOS
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from ds_utils.preprocessing.preprocessing_utilities import (
    handle_duplicates,
    handle_time_split,
    handle_time_bursts,
    handle_sensor_noise,
    handle_missing_values,
)

## handle_duplicates

`handle_duplicates` detecta timestamps duplicados, conserva la ocurrencia seleccionada (`'first'` o `'last'`) y devuelve `(df_clean, df_duplicates)`.

In [2]:
# 1. Datos sintéticos con duplicados
df_dups_raw = pd.DataFrame({
    "timestamp": [
        "2024-01-01 10:00:00",
        "2024-01-01 10:00:00",  # duplicado
        "2024-01-01 10:01:00",
        "2024-01-01 10:02:00",
        "2024-01-01 10:02:00",  # duplicado
        "2024-01-01 10:03:00",
    ],
    "value": [10.5, 10.9, 12.0, 15.1, 15.8, 14.2],
    "source": ["sensor_v1", "sensor_v2", "sensor_v1", "sensor_v1", "sensor_v2", "sensor_v1"]
})

print("Datos originales con marcas de tiempo repetidas:")
display(df_dups_raw)

# 2. Llamada a handle_duplicates conservando la primera ocurrencia
df_clean_first, df_removed_dups = handle_duplicates(
    df=df_dups_raw,
    col_time="timestamp",
    keep="first"
)

print("DataFrame limpio (keep='first'):")
display(df_clean_first)

print("Filas duplicadas descartadas:")
display(df_removed_dups)

Datos originales con marcas de tiempo repetidas:


,timestamp,value,source
0,2024-01-01 10:00:00,10.5,sensor_v1
1,2024-01-01 10:00:00,10.9,sensor_v2
2,2024-01-01 10:01:00,12.0,sensor_v1
3,2024-01-01 10:02:00,15.1,sensor_v1
4,2024-01-01 10:02:00,15.8,sensor_v2
5,2024-01-01 10:03:00,14.2,sensor_v1


DataFrame limpio (keep='first'):


,timestamp,value,source
0,2024-01-01 10:00:00,10.5,sensor_v1
1,2024-01-01 10:01:00,12.0,sensor_v1
2,2024-01-01 10:02:00,15.1,sensor_v1
3,2024-01-01 10:03:00,14.2,sensor_v1


Filas duplicadas descartadas:


,timestamp,value,source
0,2024-01-01 10:00:00,10.9,sensor_v2
1,2024-01-01 10:02:00,15.8,sensor_v2


## handle_time_split

`handle_time_split` divide un DataFrame temporal en dos conjuntos a partir de una fecha de corte (`cutoff_date`), permitiendo retener las filas anteriores o posteriores mediante `mode='tail'` o `mode='head'`.

In [3]:
# 1. Serie temporal de 5 días cada 12 horas
dates = pd.date_range("2024-01-01", "2024-01-05", freq="12h")
df_split_raw = pd.DataFrame({
    "datetime": dates,
    "reading": range(1, len(dates) + 1)
})

# 2. Partición tail con corte en 2024-01-03 00:00:00
df_main_tail, df_removed_tail = handle_time_split(
    df=df_split_raw,
    col_time="datetime",
    cutoff_date="2024-01-03 00:00:00",
    mode="tail"
)

print("DataFrame principal (hasta el corte, tail):")
display(df_main_tail)

print("Registros posteriores eliminados:")
display(df_removed_tail)

DataFrame principal (hasta el corte, tail):


,datetime,reading
0,2024-01-01 00:00:00,1
1,2024-01-01 12:00:00,2
2,2024-01-02 00:00:00,3
3,2024-01-02 12:00:00,4
4,2024-01-03 00:00:00,5
5,2024-01-03 12:00:00,6


Registros posteriores eliminados:


,datetime,reading
0,2024-01-04 00:00:00,7
1,2024-01-04 12:00:00,8
2,2024-01-05 00:00:00,9


## handle_time_bursts

`handle_time_bursts` filtra eventos que ocurren en ráfagas de frecuencia superior al umbral mínimo permitido (`freq`). Conserva el primer registro de la ráfaga y descarta los eventos posteriores demasiado próximos en el tiempo.

In [4]:
# 1. Timestamps con ráfagas a intervalos de pocos segundos
df_bursts_raw = pd.DataFrame({
    "event_time": [
        "2024-01-01 10:00:00",
        "2024-01-01 10:00:10",  # ráfaga (< 1 min)
        "2024-01-01 10:00:25",  # ráfaga (< 1 min)
        "2024-01-01 10:01:05",  # válido (> 1 min)
        "2024-01-01 10:01:15",  # ráfaga (< 1 min)
        "2024-01-01 10:02:30",  # válido (> 1 min)
    ],
    "reading": [100, 102, 101, 105, 104, 110]
})

# 2. Filtrado de ráfagas con freq='1min'
df_no_burst, df_burst_deleted = handle_time_bursts(
    df=df_bursts_raw,
    col_time="event_time",
    freq="1min"
)

print("Eventos retenidos sin ráfagas:")
display(df_no_burst)

print("Eventos descartados por pertenecer a una ráfaga:")
display(df_burst_deleted)

Eventos retenidos sin ráfagas:


,event_time,reading
0,2024-01-01 10:00:00,100
1,2024-01-01 10:01:05,105
2,2024-01-01 10:02:30,110


Eventos descartados por pertenecer a una ráfaga:


,event_time,reading
0,2024-01-01 10:00:10,102
1,2024-01-01 10:00:25,101
2,2024-01-01 10:01:15,104


## handle_sensor_noise

`handle_sensor_noise` detecta picos aislados de ruido en sensores contrastando contra tolerancias configurables. Si el salto de la señal no persiste durante una ventana futura (`window`), se corrige reemplazándolo por el valor limpio anterior y generando flags booleanas de ruido.

In [5]:
# 1. Señal con un pico aislado (índice 2) y un escalón real (índice 5)
df_noise_raw = pd.DataFrame({
    "timestamp": pd.date_range("2024-01-01 00:00:00", periods=8, freq="1min"),
    "temperature": [20.0, 20.2, 95.0, 20.3, 20.1, 50.0, 50.2, 50.1],
    "pressure":    [1.0, 1.0, 1.0, 1.1, 1.0, 1.0, 1.0, 1.0]
})

tolerances = {"temperature": 10.0}

# 2. Detección y corrección con features adicionales para ML
df_noise_clean, df_noise_rows = handle_sensor_noise(
    df=df_noise_raw,
    tolerances=tolerances,
    window=3,
    add_extra_features=True
)

print("Señal limpia y variables generadas:")
display(df_noise_clean[["timestamp", "temperature", "temperature_clean", "temperature_is_noise", "any_sensor_noise", "temperature_noise_magnitude"]])

print("Filas clasificadas como ruidosas:")
display(df_noise_rows)

Señal limpia y variables generadas:


,timestamp,temperature,temperature_clean,temperature_is_noise,any_sensor_noise,temperature_noise_magnitude
0,2024-01-01 00:00:00,20.0,20.0,False,False,0.0
1,2024-01-01 00:01:00,20.2,20.2,False,False,0.0
2,2024-01-01 00:02:00,95.0,20.2,True,True,74.8
3,2024-01-01 00:03:00,20.3,20.3,False,False,0.0
4,2024-01-01 00:04:00,20.1,20.1,False,False,0.0
5,2024-01-01 00:05:00,50.0,50.0,False,False,0.0
6,2024-01-01 00:06:00,50.2,50.2,False,False,0.0
7,2024-01-01 00:07:00,50.1,50.1,False,False,0.0


Filas clasificadas como ruidosas:


,timestamp,temperature,pressure,temperature_clean,temperature_is_noise,temperature_diff,temperature_diff_clean,temperature_noise_magnitude,temperature_rolling_std,any_sensor_noise
0,2024-01-01 00:02:00,95.0,1.0,20.2,True,74.8,0.0,74.8,0.11547,True


## handle_missing_values

`handle_missing_values` audita los valores ausentes en el DataFrame y permite imputarlos selectivamente mediante un valor escalar o un diccionario de reemplazos específicos por columna.

In [7]:
# 1. DataFrame con valores nulos heterogéneos
df_na_raw = pd.DataFrame({
    "user_id": [101, 102, 103, 104],
    "age": [25.0, np.nan, 30.0, np.nan],
    "city": ["Madrid", "Barcelona", None, "Valencia"],
    "score": [8.5, 9.0, 7.5, 6.0]
})

# 2. Auditoría e imputación con diccionario
fill_spec = {
    "age": 0.0,
    "city": "Desconocida"
}

df_na_imputed, df_audit = handle_missing_values(
    df=df_na_raw,
    fill_value=fill_spec
)

print("DataFrame tras imputación:")
display(df_na_imputed)

print("Tabla de auditoría de nulos:")
display(df_audit)

DataFrame tras imputación:


,user_id,age,city,score
0,101,25.0,Madrid,8.5
1,102,0.0,Barcelona,9.0
2,103,30.0,Desconocida,7.5
3,104,0.0,Valencia,6.0


Tabla de auditoría de nulos:


,column,n_missing,dtype,fill_value
0,age,2,float64,0.0
1,city,1,str,Desconocida
